# Tier 2 Assignment 2 — Solutions: Structure and Function

Complete solutions for all 7 problems.


In [ ]:
from Bio import PDB
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import math

## Problem 1: PDB Parser

Use `PDBParser` to load the structure. Iterate chains, then residues. Skip
hetero-residues (water, ligands) by checking `residue.id[0] == ' '`.

In [ ]:
def parse_pdb(pdb_path: str) -> dict:
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure("structure", pdb_path)

    # Use the first model
    model = next(structure.get_models())

    residues_per_chain = {}
    unique_residue_types = set()

    for chain in model:
        std_residues = [
            res for res in chain
            if res.id[0] == ' '  # ATOM record (standard residue)
        ]
        residues_per_chain[chain.id] = len(std_residues)
        for res in std_residues:
            unique_residue_types.add(res.resname.strip())

    return {
        "num_chains": len(residues_per_chain),
        "residues_per_chain": residues_per_chain,
        "unique_residue_types": sorted(unique_residue_types),
    }


# Demo (requires a PDB file)
# from Bio.PDB import PDBList
# pdbl = PDBList()
# pdbl.retrieve_pdb_file('1crn', pdir='.', file_type='pdb')
# result = parse_pdb('./pdb1crn.ent')
# for key, value in result.items():
#     print(f"{key}: {value}")
print("parse_pdb defined. Uncomment demo to run.")

## Problem 2: Protein Properties

`ProteinAnalysis` wraps many property calculations. `gravy()` gives the Grand
Average of Hydropathicity. Amino acid composition is returned as a fraction dict
by dividing each count by sequence length.

In [ ]:
def protein_properties(sequence: str) -> dict:
    analysis = ProteinAnalysis(sequence)
    raw_composition = analysis.get_amino_acids_percent()
    # get_amino_acids_percent already returns fractions (not percentages, despite the name)
    return {
        "molecular_weight": round(analysis.molecular_weight(), 2),
        "isoelectric_point": round(analysis.isoelectric_point(), 2),
        "gravy": round(analysis.gravy(), 3),
        "aa_composition": {aa: round(frac, 4) for aa, frac in raw_composition.items()},
    }


# Test
test_protein = "MKTAYIAKQRQISFVKSHFSRQ"
props = protein_properties(test_protein)
for key, value in props.items():
    if key == "aa_composition":
        non_zero = {aa: v for aa, v in value.items() if v > 0}
        print(f"{key}: {non_zero}")
    else:
        print(f"{key}: {value}")

## Problem 3: Secondary Structure Composition

Run DSSP via `Bio.PDB.DSSP`. The second element of each DSSP record is the
secondary structure code. Map to three categories then compute fractions.
If DSSP is unavailable, fall back to phi/psi angle approximation.

In [ ]:
def secondary_structure_composition(pdb_path: str, use_dssp: bool = True) -> dict:
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure("s", pdb_path)
    model = next(structure.get_models())

    if use_dssp:
        dssp = PDB.DSSP(model, pdb_path)
        # DSSP codes: H=alpha-helix, G=3-10 helix, I=pi-helix,
        #             E=beta-strand, B=isolated bridge,
        #             T=turn, S=bend, -=coil
        helix_codes = {'H', 'G', 'I'}
        sheet_codes = {'E', 'B'}

        counts = {'helix': 0, 'sheet': 0, 'coil': 0}
        for key in dssp.property_dict:
            ss = dssp[key][2]  # index 2 is the SS code
            if ss in helix_codes:
                counts['helix'] += 1
            elif ss in sheet_codes:
                counts['sheet'] += 1
            else:
                counts['coil'] += 1
    else:
        # Phi/psi approximation
        polypeptide_builder = PDB.PPBuilder()
        counts = {'helix': 0, 'sheet': 0, 'coil': 0}
        for pp in polypeptide_builder.build_peptides(model):
            phi_psi = pp.get_phi_psi_list()
            for phi, psi in phi_psi:
                if phi is None or psi is None:
                    counts['coil'] += 1
                    continue
                phi_deg = math.degrees(phi)
                psi_deg = math.degrees(psi)
                if -100 < phi_deg < -30 and -80 < psi_deg < -10:
                    counts['helix'] += 1
                elif phi_deg < -90 and (psi_deg > 90 or psi_deg < -150):
                    counts['sheet'] += 1
                else:
                    counts['coil'] += 1

    total = sum(counts.values())
    if total == 0:
        return {'helix': 0.0, 'sheet': 0.0, 'coil': 0.0}
    return {cat: round(count / total, 4) for cat, count in counts.items()}


# Demo
# comp = secondary_structure_composition('./pdb1crn.ent', use_dssp=False)
# print(comp)
# print(f"Sum: {sum(comp.values()):.4f}")
print("secondary_structure_composition defined. Uncomment demo to run.")

## Problem 4: Restriction Site Finder

For each enzyme recognition sequence, scan the uppercased DNA with a simple
sliding-window `str.find` in a while loop. Convert to 1-based at return time.
Because all four recognition sequences here are palindromic, a single-strand scan
finds all cut sites.

In [ ]:
RESTRICTION_ENZYMES = {
    "EcoRI":   "GAATTC",
    "BamHI":   "GGATCC",
    "HindIII": "AAGCTT",
    "NotI":    "GCGGCCGC",
}


def find_restriction_sites(dna: str, enzymes: dict[str, str] | None = None) -> list[tuple]:
    if enzymes is None:
        enzymes = RESTRICTION_ENZYMES
    seq = dna.upper()
    sites = []
    for enzyme, recognition in enzymes.items():
        start = 0
        while True:
            pos = seq.find(recognition, start)
            if pos == -1:
                break
            sites.append((pos + 1, enzyme))  # 1-based
            start = pos + 1
    return sorted(sites)


# Test
test_dna = "AAAGAATTCGGGGATCCTTAAGCTTCCCGCGGCCGCAAA"
sites = find_restriction_sites(test_dna)
for pos, enzyme in sites:
    print(f"Position {pos}: {enzyme}")
# Expected:
# Position 4:  EcoRI   (GAATTC starts at index 3, 1-based = 4)
# Position 10: BamHI
# Position 19: HindIII (AAGCTT — note overlapping with the BamHI GGATCC complement)
# Position 30: NotI

## Problem 5: Motif Scanner

At each position, look up the column score from the PWM for the nucleotide at
that position. Sum all columns. Report positions on the reverse complement with
a negative sign. Use `str.maketrans` for the reverse complement.

In [ ]:
TATA_PWM = {
    'A': [-1.5,  1.8, -1.5,  1.8,  1.5,  1.5,  1.2,  1.2],
    'C': [-1.5, -1.5, -1.5, -1.5, -1.5, -1.5, -1.5, -1.5],
    'G': [-1.5, -1.5, -1.5, -1.5, -1.5, -1.5, -1.5, -1.5],
    'T': [ 1.8, -1.5,  1.8, -1.5, -1.5, -1.5, -1.5, -1.5],
}

_COMP = str.maketrans('ACGT', 'TGCA')


def _score_at(dna: str, pos: int, pwm: dict[str, list[float]]) -> float:
    width = len(next(iter(pwm.values())))
    return sum(
        pwm.get(dna[pos + j], [-1.5] * width)[j]
        for j in range(width)
    )


def scan_pwm(
    dna: str,
    pwm: dict[str, list[float]],
    threshold: float,
    both_strands: bool = True,
) -> list[tuple]:
    seq = dna.upper()
    width = len(next(iter(pwm.values())))
    hits = []

    for i in range(len(seq) - width + 1):
        score = _score_at(seq, i, pwm)
        if score >= threshold:
            hits.append((i + 1, round(score, 4)))  # positive = forward strand

    if both_strands:
        rev_comp = seq.translate(_COMP)[::-1]
        for i in range(len(rev_comp) - width + 1):
            score = _score_at(rev_comp, i, pwm)
            if score >= threshold:
                # Convert rc position back to forward-strand 1-based
                fwd_pos = len(seq) - i - width + 1
                hits.append((-fwd_pos, round(score, 4)))  # negative = reverse strand

    return sorted(hits, key=lambda x: x[1], reverse=True)


# Test
test_seq = "GCGCGCGCTATAAAAGCGCGCGCTATAAAAGCGCGC"
hits = scan_pwm(test_seq, TATA_PWM, threshold=5.0)
print(f"Hits above threshold 5.0:")
for pos, score in hits:
    strand = "+" if pos > 0 else "-"
    print(f"  pos {abs(pos)} ({strand} strand): score {score:.2f}")

## Problem 6: GO Term Enrichment

Build sets of genes per GO term, then for each term construct the 2x2 table and
call `scipy.stats.fisher_exact`. Use `alternative='greater'` for one-sided test
(enrichment, not depletion).

In [ ]:
def go_enrichment(
    hit_genes: list[str],
    background_genes: list[str],
    go_annotations: dict[str, list[str]],
    p_cutoff: float = 0.05,
) -> list[tuple]:
    from scipy.stats import fisher_exact

    hit_set = set(hit_genes)
    bg_set = set(background_genes)
    n_hit = len(hit_set)
    n_bg = len(bg_set)

    # Collect all GO terms and the gene sets associated with each
    term_to_genes: dict[str, set] = {}
    for gene, terms in go_annotations.items():
        if gene not in bg_set:
            continue
        for term in terms:
            term_to_genes.setdefault(term, set()).add(gene)

    results = []
    for term, genes_with_term in term_to_genes.items():
        a = len(hit_set & genes_with_term)          # hit AND has term
        b = len(genes_with_term - hit_set)           # not hit AND has term
        c = n_hit - a                                # hit AND no term
        d = n_bg - n_hit - b                         # not hit AND no term

        odds_ratio, p_value = fisher_exact([[a, b], [c, d]], alternative='greater')
        if p_value < p_cutoff:
            results.append((term, p_value, odds_ratio, a, a + b))

    return sorted(results, key=lambda x: x[1])


# Test
background = [f"gene{i}" for i in range(1, 21)]
hits = [f"gene{i}" for i in range(1, 9)]

annotations = {}
for i in range(1, 21):
    gene = f"gene{i}"
    terms = []
    if i <= 10:
        terms.append("GO:0001")
    else:
        terms.append("GO:0002")
    if i % 2 == 0:
        terms.append("GO:0003")
    annotations[gene] = terms

enriched = go_enrichment(hits, background, annotations)
print("Enriched GO terms (p < 0.05):")
for term, pval, or_, hit_n, bg_n in enriched:
    print(f"  {term}: p={pval:.4f}, OR={or_:.2f}, hits={hit_n}/{len(hits)}, background={bg_n}/{len(background)}")

## Problem 7: Synteny Block Finder

Build a positional lookup for genome2. For each gene in genome1 that also appears
in genome2, record its genome2 position. Then find maximal consecutive runs where
the positions are strictly increasing (same-strand) or strictly decreasing
(inverted). Emit only runs of length >= min_block_size.

In [ ]:
def find_synteny_blocks(
    genome1: list[str],
    genome2: list[str],
    min_block_size: int = 3,
) -> list[dict]:
    # Map gene name to its position in genome2
    g2_pos = {gene: i for i, gene in enumerate(genome2)}

    # Build the mapping: genome1 index -> genome2 index (for shared genes)
    shared = [(i, g2_pos[gene]) for i, gene in enumerate(genome1) if gene in g2_pos]

    if len(shared) < min_block_size:
        return []

    blocks = []

    def emit_block(run: list[tuple], orientation: str) -> None:
        if len(run) < min_block_size:
            return
        g1_idx = [p[0] for p in run]
        g2_idx = [p[1] for p in run]
        genes = [genome1[i] for i in g1_idx]
        blocks.append({
            "genes": genes,
            "genome1_positions": g1_idx,
            "genome2_positions": g2_idx,
            "orientation": orientation,
        })

    # Extend runs greedily
    i = 0
    while i < len(shared):
        # Try to extend a same-strand run from position i
        same_run = [shared[i]]
        j = i + 1
        while j < len(shared):
            g1_prev, g2_prev = same_run[-1]
            g1_cur, g2_cur = shared[j]
            if g1_cur == g1_prev + 1 and g2_cur == g2_prev + 1:
                same_run.append(shared[j])
                j += 1
            else:
                break

        # Try to extend an inverted run from position i
        inv_run = [shared[i]]
        k = i + 1
        while k < len(shared):
            g1_prev, g2_prev = inv_run[-1]
            g1_cur, g2_cur = shared[k]
            if g1_cur == g1_prev + 1 and g2_cur == g2_prev - 1:
                inv_run.append(shared[k])
                k += 1
            else:
                break

        # Keep the longer run; must meet min_block_size
        best_run, orientation, advance = max(
            [(same_run, 'same', j), (inv_run, 'inverted', k)],
            key=lambda x: len(x[0]),
        )

        if len(best_run) >= min_block_size:
            emit_block(best_run, orientation)
            i = advance
        else:
            i += 1

    return blocks


# Test
g1 = ["geneA", "geneB", "geneC", "geneD", "geneE", "geneF", "geneG", "geneH"]
g2 = ["geneC", "geneD", "geneE", "geneX", "geneH", "geneG", "geneF", "geneA"]

blocks = find_synteny_blocks(g1, g2, min_block_size=3)
print(f"Found {len(blocks)} synteny block(s):")
for b in blocks:
    print(f"  {b['orientation'].upper():8s} | genes: {b['genes']}")
    print(f"             g1 positions: {b['genome1_positions']}")
    print(f"             g2 positions: {b['genome2_positions']}")

---

## Key Patterns

1. **PDB**: `residue.id[0] == ' '` filters ATOM records; HETATM has a letter prefix
2. **ProteinAnalysis**: `get_amino_acids_percent()` returns fractions, not percentages
3. **DSSP**: index 2 of the record tuple holds the secondary-structure code
4. **Restriction sites**: scan in a `while` loop with `str.find(pattern, start)` for overlapping hits
5. **PWM scoring**: build a helper function for one-position scoring to keep `scan_pwm` readable
6. **Fisher exact**: `alternative='greater'` for enrichment; construct the 2x2 table carefully
7. **Synteny**: greedily extend same and inverted runs in parallel; keep the longer one
